# FormulaLM — Base Model Baseline on `test_real_v2` (Zero-Shot & 5-Shot)

Evaluates untouched `Qwen/Qwen3-1.7B` at the pinned revision (`70d244cc86ccca08cf5af4e1e306ecf908b1ad5e`, no adapter, fp16, non-thinking mode) on `data/test_real_v2/test_real_v2.csv` (50 examples) using the **exact same settings** as the earlier baselines (`results/baseline_*`):

1. `results/baseline_zeroshot_real_v2` (`few_shot=0`, `seed=42`, `temperature=0`, `max_tokens=96`, `batch_size=8`, `resume=True`)
2. `results/baseline_5shot_real_v2` (`few_shot=5`, drawn deterministically from `data/processed/train.jsonl` with `seed=42`, matching `baseline_5shot_real` and `baseline_5shot_synthetic`)

Then regenerates `results/final_headline.md` (and `.json`) comparing Base 0-shot, Base 5-shot, Fine-tuned v1, and Fine-tuned v2 across all three test sets with 95% Wilson CIs, and pushes results back to GitHub.

**Hardware:** Colab or Kaggle T4 GPU (~2 minutes total).

In [ ]:
# 1. Verify CUDA GPU (must be NVIDIA T4 / CUDA GPU, NOT CPU or TPU)
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch runtime to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 2. Clone / pull FormulaLM (Colab or Kaggle), install eval deps, ensure data/processed/train.jsonl exists
import getpass, os, pathlib, subprocess, sys

IS_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
REPO_DIR = pathlib.Path('/kaggle/working/FormulaLM' if IS_KAGGLE else '/content/FormulaLM')

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for _key in ('GITHUB_TOKEN', 'HF_TOKEN'):
            try:
                os.environ[_key] = _ks.get_secret(_key)
                print(f'Loaded Kaggle secret: {_key}')
            except Exception:
                pass
    except Exception as exc:
        print('kaggle_secrets unavailable:', exc)

if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
for _p in (str(REPO_DIR), str(REPO_DIR / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.environ['PYTHONPATH'] = f"{REPO_DIR}:{REPO_DIR / 'src'}"

%pip uninstall -y -q torchao
%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft "transformers>=4.51.0" "huggingface_hub>=0.30.0"
%pip install -q --no-deps -e .

# Build v1 train.jsonl if missing so 5-shot draws the exact same 5 train examples as earlier baselines.
if not pathlib.Path('data/processed/train.jsonl').is_file():
    !python -m src.data.build_dataset --version v1 --workers 2

In [ ]:
# 3. Evaluate untouched Qwen/Qwen3-1.7B (no adapter) on test_real_v2: 0-shot and 5-shot (train only)
import gc, torch
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
BASE_REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
DATASET = 'data/test_real_v2/test_real_v2.csv'
FEW_SHOT_SOURCE = 'data/processed/train.jsonl'

print(f'Loading base model (no adapter): {MODEL_ID}@{BASE_REVISION[:8]}', flush=True)
base_backend = TransformersBackend(
    MODEL_ID,
    adapter_path=None,
    revision=BASE_REVISION,
    load_in_4bit=False,
    enable_thinking=False,
)

for run_name, few_shot in [
    ('baseline_zeroshot_real_v2', 0),
    ('baseline_5shot_real_v2', 5),
]:
    print(f'=== Running {run_name} (few_shot={few_shot}) ===', flush=True)
    m = evaluate(
        backend=base_backend,
        dataset_path=DATASET,
        run_name=run_name,
        seed=42,
        max_tokens=96,
        few_shot=few_shot,
        few_shot_source=FEW_SHOT_SOURCE,
        batch_size=8,
        resume=True,
    )
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | avg_lat={ov['avg_latency_ms']:.1f}ms",
        flush=True,
    )

del base_backend
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 4. Regenerate unified results/final_headline.md (Base 0-shot, Base 5-shot, v1, v2 with 95% Wilson CIs)
!python -m src.eval.compare_v1_v2 \
  --results-dir results \
  --out results/ft_v2_comparison.md \
  --json-out results/ft_v2_comparison.json \
  --headline-out results/final_headline.md \
  --headline-json results/final_headline.json

In [ ]:
# 5. Push lightweight baseline results & final_headline back to GitHub
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/baseline_zeroshot_real_v2 results/baseline_5shot_real_v2 results/final_headline.md results/final_headline.json
    !git diff --cached --quiet || git commit -m "results: add Qwen3-1.7B 0-shot & 5-shot baselines on test_real_v2 and final headline table"
    !git pull --rebase origin main
    !git push origin main